# GLM-5.2 (FPT AI Factory) rate-limit check

FPT AI Factory's GLM-5.2 endpoint enforces **RPM = 50** (requests/minute) and
**TPM = 100,000** (tokens/minute). The few-shot inference loop in
`vsf-few-shot-vinumqa.ipynb` fires one request per test sample (497 total)
with no throttling -- each request also carries a large fixed few-shot
prefix (3 demonstrations, each with a full table) on top of the per-sample
context, so prompt size is the main risk for TPM, not just request count.

This notebook answers two questions before changing the main inference loop:
1. How many prompt tokens does a *real* request in this pipeline actually
   cost (measured via the API's own `usage.prompt_tokens`, not an estimate)?
2. At what request rate does that tokens/request number blow through
   TPM = 100,000, independently of the RPM = 50 cap?

Run this against a small sample (default: 15 requests) -- enough to get a
stable per-request token estimate without burning a large chunk of the
per-minute budget just to measure it.

In [1]:
%pip install -q tiktoken


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from pathlib import Path
from tabulate import tabulate

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

df = pd.read_json(PROJECT_ROOT / "datasets" / "ViNumQA" / "test.json")
len(df)


497

## Context formatting + prompt assembly (identical to `vsf-few-shot-vinumqa.ipynb`)

Reuses the same `SYSTEM_MESSAGE`, `USER_MESSAGE_FRAME`, and `FEW_SHOT_EXAMPLES`
as the main few-shot notebook, so the measured token counts here reflect the
actual production prompt shape -- not a simplified stand-in.

In [3]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

df["pre_text_processed"] = df.apply(lambda x: formatting_pre_text(x), axis=1)
df["post_text_processed"] = df.apply(lambda x: formatting_post_text(x), axis=1)
df["table_processed"] = df.apply(lambda x: formatting_table(x), axis=1)
df["input_question"] = df.apply(lambda x: x["qa"]["question"], axis=1)

df = df[["pre_text_processed", "table_processed", "post_text_processed", "input_question"]]
df.columns = [["pre_text", "table", "post_text", "question"]]
df.columns = ["pre_text", "table", "post_text", "question"]
df.head(3)


,pre_text,table,post_text,question
0,thuyết minh báo cáo tài chính hợp nhất ( tiếp ...,| các thành phần của ảnh hưởng lũy kế của việc...,.,Sự thay đổi trong thu nhập ròng từ hiệu ứng tí...
1,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,.,"Theo dự phóng, doanh thu và lợi nhuận ròng quý..."
2,"sử dụng phương pháp p/b và rnav để định giá, c...",| Năm tài chính (31/12) | 2016 | 2017 | ...,.,IDC có bao nhiêu ha quỹ đất sẵn sàng cho thuê ...


In [4]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

FEW_SHOT_EXAMPLES = [
    {
        "pre_text": "phụ lục iv ace limited và các công ty con thông tin bổ sung về phí tái bảo hiểm thu được cho các năm kết thúc ngày 31 tháng 12 năm 2010, 2009 và 2008 (tính bằng triệu đô la mỹ, ngoại trừ tỷ lệ phần trăm) số tiền trực tiếp nhượng cho các công ty nhận từ các công ty khác số tiền ròng tỷ lệ phần trăm số tiền nhận được trên.",
        "table": (
            "|   cho các năm kết thúc ngày 31 tháng 12 năm 2010, 2009 và 2008 (tính bằng triệu đô la Mỹ, ngoại trừ tỷ lệ phần trăm) | số tiền trực tiếp   | nhượng cho các công ty khác   | nhận từ các công ty khác   | số tiền ròng   | tỷ lệ phần trăm số tiền nhận được trên số tiền ròng   |\n"
            "|----------------------------------------------------------------------------------------------------------------------|---------------------|-------------------------------|----------------------------|----------------|-------------------------------------------------------|\n"
            "|                                                                                                                 2010 | $ 15780             | $ 5792                        | $ 3516                     | $ 13504        | 26% ( 26 % )                                          |\n"
            "|                                                                                                                 2009 | $ 15415             | $ 5943                        | $ 3768                     | $ 13240        | 28% ( 28 % )                                          |\n"
            "|                                                                                                                 2008 | $ 16087             | $ 6144                        | $ 3260                     | $ 13203        | 25% ( 25 % )                                          |"
        ),
        "post_text": ".",
        "question": "Sự khác biệt giữa số tiền chuyển giao và nhận chuyển giao trong năm 2010 là bao nhiêu?",
        "program": "subtract(5792, 3516)",
    },
    {
        "pre_text": "hệ số khả năng thanh toán lãi vay cũng tăng cao đạt mức 13.2 lần, so với chỉ 10.3 lần cùng kỳ.",
        "table": (
            "|                   |   FY 2015 |   FY 2016 |   FY 2017 |   FY 2018 |   FY 2019(F) |\n"
            "|-------------------|-----------|-----------|-----------|-----------|--------------|\n"
            "| Doanh thu (VNDbn) |      4920 |     11217 |     15297 |     38664 |        71115 |\n"
            "| Lãi gộp (Vbn)     |       718 |      2420 |      3128 |      7617 |        10983 |"
        ),
        "post_text": ".",
        "question": "Hệ số khả năng thanh toán lãi vay tăng bao nhiêu lần so với cùng kỳ năm ngoái?",
        "program": "subtract(13.2, 10.3)",
    },
    {
        "pre_text": "tỷ lệ phần trăm chi phí vốn trên phần trăm tổng tài sản của mảng viễn thông được duy trì trên 1, cho thấy sự tập trung phân bổ chi phí vốn vào mảng viễn thông của fpt qua các năm.\nngoài ra, tỷ lệ này của mảng đầu tư và giáo dục là 1,1 vào năm 2018 và 0,9 vào năm 2019, khẳng định fpt cũng đang tập trung vào phát triển 2 mảng này trong 2 năm gần đây.",
        "table": (
            "|                     |   2014 |   2015 |   2016 |   2017 |   2018 |   2019 |\n"
            "|---------------------|--------|--------|--------|--------|--------|--------|\n"
            "| Viễn thông          |    1.8 |    2.4 |    1.9 |    1.4 |    1.7 |    1.8 |\n"
            "| Nội dung số         |    0.4 |    0.2 |    0.9 |    0.1 |    0.1 |    0.1 |\n"
            "| Phát triển phần mềm |    2.4 |    1.3 |    3.1 |    1.1 |    0.4 |    0.5 |"
        ),
        "post_text": ".",
        "question": "Tổng tỷ lệ của mảng Nội dung số trong ba năm từ 2014 đến 2016 là bao nhiêu?",
        "program": "add(0.4, 0.2), add(#0, 0.9)",
    },
]

few_shot_messages = []
for shot in FEW_SHOT_EXAMPLES:
    few_shot_messages.append({
        "role": "user",
        "content": USER_MESSAGE_FRAME.format(
            pre_text=shot["pre_text"], table=shot["table"],
            post_text=shot["post_text"], question=shot["question"],
        )
    })
    few_shot_messages.append({"role": "assistant", "content": shot["program"]})


## Local pre-estimate (tiktoken)

`tiktoken` doesn't ship a GLM tokenizer, so this uses `cl100k_base` (GPT-4's
encoding) as a rough stand-in -- it will not match GLM's actual tokenizer
exactly, especially for Vietnamese text, but it's good enough to sanity-check
the order of magnitude and to catch outlier long samples before spending API
calls. The real per-request cost is measured in the next section via the
API's own `usage.prompt_tokens`, which is the number that actually counts
against TPM.

In [5]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

def build_messages(row):
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        *few_shot_messages,
        {
            "role": "user",
            "content": USER_MESSAGE_FRAME.format(
                pre_text=row["pre_text"], table=row["table"],
                post_text=row["post_text"], question=row["question"],
            ),
        },
    ]

def estimate_prompt_tokens(messages):
    # Rough: sum tokens of each message's content plus a small per-message
    # overhead (role tokens, separators) -- not exact, just an upper bound.
    return sum(len(enc.encode(m["content"])) for m in messages) + 4 * len(messages)

estimated = [estimate_prompt_tokens(build_messages(row)) for _, row in df.iterrows()]
import numpy as np
estimated = np.array(estimated)
print(f"n={len(estimated)}")
print(f"min={estimated.min()}  mean={estimated.mean():.0f}  median={np.median(estimated):.0f}  max={estimated.max()}")
print(f"p95={np.percentile(estimated, 95):.0f}")


n=497
min=1858  mean=3298  median=3339  max=6093
p95=4407


## Real per-request cost + RPM/TPM budget check

Sends a small number of real requests (`N_PROBE`, default 15) through the
FPT AI Factory endpoint and reads `chat_completion.usage.prompt_tokens` /
`.completion_tokens` / `.total_tokens` directly from the API response -- the
authoritative token count, not an estimate. Each request is timestamped so
actual requests/minute and tokens/minute can be checked against the
RPM=50 / TPM=100,000 limits.

Set `API_KEY` / `BASE_URL` via `.env` (same as the main notebook).

In [6]:
import os
import time
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ["API_KEY"]
BASE_URL = os.environ["BASE_URL"]
MODEL = "GLM-5.2"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

RPM_LIMIT = 50
TPM_LIMIT = 100_000
N_PROBE = 15  # keep small: this is a measurement pass, not the real run


In [7]:
probe_rows = df.sample(n=N_PROBE, random_state=0)

probe_log = []
for df_index, row in probe_rows.iterrows():
    t0 = time.monotonic()
    chat_completion = client.chat.completions.create(
        model=MODEL,
        messages=build_messages(row),
        temperature=0.0,
        max_tokens=1024,
        stream=False,
    )
    elapsed = time.monotonic() - t0

    usage = chat_completion.usage
    probe_log.append({
        "df_index": df_index,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "elapsed_s": elapsed,
    })
    print(
        f"[{len(probe_log)}/{N_PROBE}] idx={df_index}  "
        f"prompt={usage.prompt_tokens}  completion={usage.completion_tokens}  "
        f"total={usage.total_tokens}  elapsed={elapsed:.2f}s"
    )


[1/15] idx=90  prompt=1850  completion=131  total=1981  elapsed=2.02s
[2/15] idx=254  prompt=2500  completion=153  total=2653  elapsed=3.25s
[3/15] idx=283  prompt=2692  completion=173  total=2865  elapsed=2.41s
[4/15] idx=443  prompt=2425  completion=886  total=3311  elapsed=25.64s
[5/15] idx=336  prompt=1927  completion=188  total=2115  elapsed=2.67s
[6/15] idx=15  prompt=2736  completion=158  total=2894  elapsed=2.17s
[7/15] idx=316  prompt=2628  completion=60  total=2688  elapsed=1.14s
[8/15] idx=486  prompt=2532  completion=152  total=2684  elapsed=2.11s
[9/15] idx=159  prompt=3106  completion=250  total=3356  elapsed=3.09s
[10/15] idx=153  prompt=2071  completion=246  total=2317  elapsed=3.06s
[11/15] idx=241  prompt=2660  completion=745  total=3405  elapsed=10.64s
[12/15] idx=250  prompt=1989  completion=615  total=2604  elapsed=8.70s
[13/15] idx=426  prompt=2834  completion=111  total=2945  elapsed=2.73s
[14/15] idx=289  prompt=2175  completion=120  total=2295  elapsed=4.09s
[1

## Does GLM-5.2 return separate reasoning content?

GLM-5.2 is a reasoning model -- many reasoning-model APIs (DeepSeek-R1,
GLM-Zero style) return the chain-of-thought in a separate
`message.reasoning_content` field, distinct from `message.content` (the final
answer). If FPT AI Factory's endpoint does the same:
- `usage.completion_tokens` almost certainly already includes reasoning
  tokens (it's the server's real generation cost), so the TPM measurement
  above is unaffected either way.
- But the main loop's `output = chat_completion.choices[0].message.content...`
  needs checking: if reasoning leaks into `content` instead of being cleanly
  separated, the program string gets polluted with prose the scorer's
  `extract_program` has to strip out.

This makes exactly one real request and dumps the full `message` object to
see which fields are populated.

In [11]:
probe_row = df.sample(n=1, random_state=1).iloc[0]

probe_completion = client.chat.completions.create(
    model=MODEL,
    messages=build_messages(probe_row),
    temperature=0.0,
    max_tokens=2048,
    stream=False,
)

message = probe_completion.choices[0].message

print("Fields on message object:", message.model_fields_keys() if hasattr(message, "model_fields_keys") else list(message.model_dump().keys()))
print()
print("=== message.content ===")
print(repr(message.content))
print()
reasoning = getattr(message, "reasoning_content", None)
print("=== message.reasoning_content ===")
print(repr(reasoning))
print()
print("=== full message.model_dump() ===")
import json as _json
print(_json.dumps(message.model_dump(), ensure_ascii=False, indent=2))
print()
print("=== usage ===")
print(probe_completion.usage.model_dump())


Fields on message object: ['content', 'refusal', 'role', 'audio', 'function_call', 'tool_calls', 'reasoning_content']

=== message.content ===
'add(2408, 1364)'

=== message.reasoning_content ===
'The question asks for the total purchase commitments for the next two years in thousands.\n\nLooking at the table:\n- 2006: $2408\n- 2007: $1364\n- Total: $3772\n\nThe question asks for the total, which is already given in the table as $3772. But I need to compute it using the operators.\n\nI can add 2408 and 1364 to get the total.'

=== full message.model_dump() ===
{
  "content": "add(2408, 1364)",
  "refusal": null,
  "role": "assistant",
  "audio": null,
  "function_call": null,
  "tool_calls": null,
  "reasoning_content": "The question asks for the total purchase commitments for the next two years in thousands.\n\nLooking at the table:\n- 2006: $2408\n- 2007: $1364\n- Total: $3772\n\nThe question asks for the total, which is already given in the table as $3772. But I need to compute it u

In [8]:
probe_df = pd.DataFrame(probe_log)

mean_prompt = probe_df["prompt_tokens"].mean()
mean_total = probe_df["total_tokens"].mean()
mean_elapsed = probe_df["elapsed_s"].mean()

print(f"Measured over {len(probe_df)} real requests (cl100k_base estimate above was {estimated.mean():.0f} for comparison):")
print(f"  mean prompt_tokens  = {mean_prompt:.0f}")
print(f"  mean total_tokens   = {mean_total:.0f}  (prompt + completion)")
print(f"  mean elapsed/request = {mean_elapsed:.2f}s")
print()

# TPM check: how many requests/minute can this prompt size sustain before
# hitting TPM=100,000, independent of the RPM=50 cap?
max_requests_per_min_by_tpm = TPM_LIMIT / mean_total
print(f"TPM budget allows at most {max_requests_per_min_by_tpm:.1f} requests/min at this token size "
      f"(TPM={TPM_LIMIT} / {mean_total:.0f} tokens/request).")
print(f"RPM limit is {RPM_LIMIT} requests/min.")

if max_requests_per_min_by_tpm < RPM_LIMIT:
    print(f"\n=> TPM is the binding constraint (not RPM): "
          f"safe max ~{max_requests_per_min_by_tpm:.1f} req/min, i.e. one request every "
          f"{60 / max_requests_per_min_by_tpm:.2f}s.")
else:
    print(f"\n=> RPM is the binding constraint (not TPM): "
          f"safe max {RPM_LIMIT} req/min, i.e. one request every {60 / RPM_LIMIT:.2f}s.")


Measured over 15 real requests (cl100k_base estimate above was 3298 for comparison):
  mean prompt_tokens  = 2407
  mean total_tokens   = 2721  (prompt + completion)
  mean elapsed/request = 6.05s

TPM budget allows at most 36.7 requests/min at this token size (TPM=100000 / 2721 tokens/request).
RPM limit is 50 requests/min.

=> TPM is the binding constraint (not RPM): safe max ~36.7 req/min, i.e. one request every 1.63s.


## Sequential-loop time estimate

With the measured per-request token/time cost, project how long a full,
naive sequential run over all 497 test samples would take under each
scenario -- unthrottled (current code, will likely 429/error once limits are
hit) vs. throttled to the binding constraint found above.

In [9]:
N_SAMPLES = len(df)

binding_req_per_min = min(RPM_LIMIT, max_requests_per_min_by_tpm)
min_interval_s = 60 / binding_req_per_min

projected_minutes_throttled = N_SAMPLES / binding_req_per_min
projected_minutes_unthrottled = N_SAMPLES * mean_elapsed / 60

print(f"Full run: {N_SAMPLES} samples")
print(f"  Unthrottled, back-to-back at measured {mean_elapsed:.2f}s/request: "
      f"~{projected_minutes_unthrottled:.1f} min IF no rate-limit errors occurred "
      f"(they will, once >{RPM_LIMIT} requests land in any 60s window or "
      f"cumulative tokens exceed {TPM_LIMIT}/min).")
print(f"  Throttled to the binding limit ({binding_req_per_min:.1f} req/min, "
      f"i.e. >= {min_interval_s:.2f}s between requests): ~{projected_minutes_throttled:.1f} min.")


Full run: 497 samples
  Unthrottled, back-to-back at measured 6.05s/request: ~50.1 min IF no rate-limit errors occurred (they will, once >50 requests land in any 60s window or cumulative tokens exceed 100000/min).
  Throttled to the binding limit (36.7 req/min, i.e. >= 1.63s between requests): ~13.5 min.


## Throttling helper for the main inference loop

Drop-in rate limiter for `vsf-few-shot-vinumqa.ipynb`'s generation loop: a
sliding-window tracker over the last 60 seconds that blocks (via `time.sleep`)
before issuing a request if either the request count or the cumulative token
count in that window would exceed the FPT AI Factory limits. Token count per
request is estimated *before* the call (from the prompt) since the real
`usage.prompt_tokens` isn't known until after the response arrives; the
window is updated with the real `usage.total_tokens` afterward so the TPM
accounting stays accurate for subsequent requests.

In [10]:
from collections import deque

class RateLimiter:
    """Sliding 60s window limiter for RPM and TPM.

    Call `wait_if_needed(estimated_tokens)` before each request (blocks via
    time.sleep if the window is already at capacity), then `record(actual_tokens)`
    after the response arrives with the real usage.total_tokens.
    """

    def __init__(self, rpm_limit: int, tpm_limit: int, window_s: float = 60.0):
        self.rpm_limit = rpm_limit
        self.tpm_limit = tpm_limit
        self.window_s = window_s
        self.request_times = deque()   # timestamps of requests in the window
        self.token_events = deque()    # (timestamp, tokens) in the window

    def _prune(self, now: float):
        while self.request_times and now - self.request_times[0] > self.window_s:
            self.request_times.popleft()
        while self.token_events and now - self.token_events[0][0] > self.window_s:
            self.token_events.popleft()

    def wait_if_needed(self, estimated_tokens: int):
        while True:
            now = time.monotonic()
            self._prune(now)

            tokens_in_window = sum(t for _, t in self.token_events)
            requests_in_window = len(self.request_times)

            over_rpm = requests_in_window >= self.rpm_limit
            over_tpm = tokens_in_window + estimated_tokens > self.tpm_limit

            if not over_rpm and not over_tpm:
                return

            # Sleep until the oldest event in the window falls out of it.
            oldest = min(
                self.request_times[0] if self.request_times else float("inf"),
                self.token_events[0][0] if self.token_events else float("inf"),
            )
            sleep_for = max(0.05, self.window_s - (now - oldest))
            time.sleep(sleep_for)

    def record(self, actual_tokens: int):
        now = time.monotonic()
        self.request_times.append(now)
        self.token_events.append((now, actual_tokens))


limiter = RateLimiter(rpm_limit=RPM_LIMIT, tpm_limit=TPM_LIMIT)

# Usage in the main generation loop (replace the bare `client.chat.completions.create(...)` call):
#
#   for df_index, values in tqdm(df.iterrows(), total=len(df)):
#       messages = [...]
#       est_tokens = estimate_prompt_tokens(messages) + 1024  # + max_tokens headroom
#       limiter.wait_if_needed(est_tokens)
#
#       chat_completion = client.chat.completions.create(
#           model=MODEL, messages=messages, temperature=0.0, max_tokens=1024, stream=False,
#       )
#       limiter.record(chat_completion.usage.total_tokens)
#
#       output = chat_completion.choices[0].message.content.strip().strip("\n")
#       df.at[df_index, "generated_program"] = output
print("RateLimiter ready -- see the usage sketch in this cell's comment for wiring into the main loop.")


RateLimiter ready -- see the usage sketch in this cell's comment for wiring into the main loop.
